<a href="https://colab.research.google.com/github/DmitriyKolesnikM8O/MOEX-Scripts/blob/main/Scripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EMA50 & EMA200

In [ ]:
# ===========================================
# Поиск акций MOEX возле EMA50 (ЧАСОВОЙ ТАЙМФРЕЙМ)
# Исправлено: живые цены, часовой пояс МСК, отсев незакрытого бара
# ===========================================

import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from tqdm import tqdm
import time

# -------------------------------------------
# Настройки
# -------------------------------------------

DISTANCE = 1.0          # Максимальное расстояние до EMA50 в %
INTERVAL = 60           # 60 минут (часовой график)
DELAY = 0.05            # Задержка между запросами

# Фиксируем часовой пояс МСК (UTC+3), чтобы избежать сдвигов на серверах/Colab
MSK_TZ = timezone(timedelta(hours=3))

# -------------------------------------------
# 1. Быстрый снимок рынка (Живые цены)
# -------------------------------------------
print("📊 Загружаем снимок рынка (живые цены прямо сейчас)...")

def get_market_snapshot():
    # Используем запрос из большого скрипта для получения цены LAST
    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {
        "iss.meta": "off",
        "iss.only": "securities,marketdata",
        "securities.columns": "SECID,SHORTNAME",
        "marketdata.columns": "SECID,LAST,VALTODAY",
    }
    r = requests.get(url, params=params, timeout=15)
    js = r.json()

    sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
    md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])

    df = sec.merge(md, on="SECID", how="left")
    df["LAST"] = pd.to_numeric(df["LAST"], errors="coerce")
    df["VALTODAY"] = pd.to_numeric(df["VALTODAY"], errors="coerce")

    # Оставляем только те акции, по которым сегодня есть торги и известна цена
    df = df[(df["VALTODAY"] > 0) & (df["LAST"].notna())].reset_index(drop=True)
    return df

stocks = get_market_snapshot()
print(f"✅ Найдено ликвидных акций с живыми ценами: {len(stocks)}")

# -------------------------------------------
# 2. Функция загрузки свечей
# -------------------------------------------
def fetch_candles(ticker, days_back=14, interval=60):
    till = datetime.now(MSK_TZ)
    since = till - timedelta(days=days_back)

    # Используем board TQBR для получения самых точных данных
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"

    all_rows, start, columns = [], 0, None
    while True:
        params = {
            "from": since.strftime("%Y-%m-%d %H:%M:%S"),
            "till": till.strftime("%Y-%m-%d %H:%M:%S"),
            "interval": interval,
            "start": start,
            "iss.meta": "off"
        }
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200:
            break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None:
            columns = js.get("columns", [])
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 500:
            break
        start += len(rows)

    if not all_rows or columns is None:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.rename(columns={
        "open": "Open", "close": "Close", "high": "High",
        "low": "Low", "volume": "Volume", "begin": "Date", "end": "DateEnd"
    })
    df = df[["Date", "DateEnd", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date")
    df = df.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)

    # ОТСЕВ НЕЗАКРЫТОГО БАРА (логика из большого скрипта)
    # Сравниваем время конца свечи с текущим МСК временем
    now_msk = pd.Timestamp.now(tz=MSK_TZ).tz_localize(None)
    if len(df) > 0 and df["DateEnd"].iloc[-1] > now_msk:
        df = df.iloc[:-1] # Убираем последнюю свечу, т.к. она еще в процессе

    return df.drop(columns=["DateEnd"]).reset_index(drop=True)

# -------------------------------------------
# 3. Основной цикл
# -------------------------------------------
results = []
print("\n🔍 Начинаем анализ акций (сравнение живой цены с EMA50)...\n")

for index, row in tqdm(stocks.iterrows(), total=len(stocks), desc="Обработка акций"):
    ticker = row["SECID"]
    name = row["SHORTNAME"]
    current_live_price = row["LAST"] # Берем цену ПРЯМО СЕЙЧАС из снипка

    try:
        df = fetch_candles(ticker, days_back=14, interval=INTERVAL)
        if df.empty or len(df) < 50:
            continue

        # Считаем EMA50 по закрытым свечам
        ema50 = df["Close"].ewm(span=50, adjust=False).mean().iloc[-1]

        # Считаем дистанцию между ЖИВОЙ ценой и актуальной EMA50
        distance = abs(current_live_price - ema50) / ema50 * 100

        if distance <= DISTANCE:
            results.append({
                "Тикер": ticker,
                "Компания": name,
                "Цена (LAST)": round(current_live_price, 4),
                "EMA50": round(ema50, 4),
                "Расстояние %": round(distance, 2),
                "Выше EMA": "✅" if current_live_price > ema50 else "❌",
            })

        time.sleep(DELAY)

    except Exception as e:
        continue

# -------------------------------------------
# 4. Вывод
# -------------------------------------------
print("\n" + "="*80)

if results:
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("Расстояние %")

    print(f"✅ НАЙДЕНО АКЦИЙ ВОЗЛЕ EMA50: {len(result_df)}")
    print("="*80)
    print("\n📊 Результаты:\n")
    print(result_df.to_string(index=False))

    filename = f"акции_возле_ema50_{datetime.now(MSK_TZ).strftime('%Y%m%d_%H%M')}.csv"
    result_df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n💾 Сохранен в: {filename}")
else:
    print("❌ Акций возле EMA50 не найдено.")

print("="*80)

📊 Загружаем снимок рынка (живые цены прямо сейчас)...
✅ Найдено ликвидных акций с живыми ценами: 276

🔍 Начинаем анализ акций (сравнение живой цены с EMA50)...



Обработка акций: 100%|██████████| 276/276 [03:19<00:00,  1.38it/s]


✅ НАЙДЕНО АКЦИЙ ВОЗЛЕ EMA50: 136

📊 Результаты:

 Тикер   Компания  Цена (LAST)      EMA50  Расстояние % Выше EMA
 SVETP Светофор п      17.0000    17.0010          0.01        ❌
  AMFL   AMFL ETF     139.0000   138.9742          0.02        ✅
 KZOSP ОргСинт ап      15.5500    15.5546          0.03        ❌
  VSMO ВСМПО-АВСМ   25120.0000 25111.9822          0.03        ✅
  RAGR    Русагро      92.0200    92.0606          0.04        ❌
  AKFB   AKFB ETF     139.0900   139.0335          0.04        ✅
  ASTR  iАстра ао     225.5000   225.6079          0.05        ❌
  CASH   CASH ETF      13.5190    13.5128          0.05        ✅
  LQDT   LQDT ETF       2.0596     2.0583          0.06        ✅
  AKMM   AKMM ETF     175.3500   175.2378          0.06        ✅
  AKMP   AKMP ETF       1.2513     1.2506          0.06        ✅
  PSMM   PSMM ETF      14.4810    14.4721          0.06        ✅
  FESH    ДВМП ао      61.7600    61.7223          0.06        ✅
  AMNR   AMNR ETF     154.8130   154.719

In [ ]:
# ==============================================================================
# ЖИВОЙ СИГНАЛЬНЫЙ СКАЧИК MOEX (Жесткая версия — Только физическое касание EMA50)
# ==============================================================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

MSK_TZ = timezone(timedelta(hours=3))
INTERVAL = 60           # 60 минут (часовой график)
DAYS_BACK = 60          # 60 дней истории

PROVEN_STOCKS = {
    "YDEX":  {"Name": "Яндекс", "Mode": "1. Макс Профит", "RR": 3.0, "Trail": 0.0, "BE": 0.0, "Note": "Жесткий тейк 1:3. Не трогать стоп руками!"},
    "UGLD":  {"Name": "ЮГК", "Mode": "3. Умеренный", "RR": 2.0, "Trail": 1.0, "BE": 1.0, "Note": "Тейк 1:2. Б/У при +1R, Трейлинг 1.0R"},
    "MTLR":  {"Name": "Мечел ао", "Mode": "1. Макс Профит", "RR": 3.0, "Trail": 0.0, "BE": 0.0, "Note": "Жесткий тейк 1:3. Держим тренд!"},
    "LSNGP": {"Name": "РСетиЛЭ-п", "Mode": "3. Умеренный", "RR": 2.0, "Trail": 1.0, "BE": 1.0, "Note": "Тейк 1:2. Б/У при +1R, Трейлинг 1.0R"},
    "HYDR":  {"Name": "РусГидро", "Mode": "3. Умеренный", "RR": 2.0, "Trail": 1.0, "BE": 1.0, "Note": "Тейк 1:2. Б/У при +1R, Трейлинг 1.0R"},
    "SNGSP": {"Name": "Сургнфгз-п", "Mode": "2. Трейлинг", "RR": 3.0, "Trail": 1.0, "BE": 0.0, "Note": "Тейк 1:3. Включать Трейлинг 1.0R"},
    "VKCO":  {"Name": "ВК", "Mode": "1. Макс Профит", "RR": 3.0, "Trail": 0.0, "BE": 0.0, "Note": "Жесткий тейк 1:3. Без трейлинга."},
    "TATN":  {"Name": "Татнефть ао", "Mode": "1. Макс Профит", "RR": 3.0, "Trail": 0.0, "BE": 0.0, "Note": "Жесткий тейк 1:3. Без трейлинга."},
    "ABIO":  {"Name": "iАРТГЕН ао", "Mode": "2. Трейлинг", "RR": 3.0, "Trail": 1.0, "BE": 0.0, "Note": "Тейк 1:3. Трейлинг 1.0R (Винрейт 60%)"},
    "SBER":  {"Name": "Сбербанк", "Mode": "2. Трейлинг", "RR": 3.0, "Trail": 1.0, "BE": 0.0, "Note": "Тейк 1:3. Трейлинг 1.0R (Винрейт 66.7%!)"},
}

def smart_round(val):
    if val >= 100: return round(val, 2)
    elif val >= 1: return round(val, 3)
    else: return round(val, 5)

def get_market_snapshot():
    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {"iss.meta": "off", "iss.only": "securities,marketdata", "securities.columns": "SECID,SHORTNAME", "marketdata.columns": "SECID,LAST,VALTODAY"}
    r = requests.get(url, params=params, timeout=15)
    js = r.json()
    sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
    md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])
    df = sec.merge(md, on="SECID", how="left")
    df["LAST"] = pd.to_numeric(df["LAST"], errors="coerce")
    df["VALTODAY"] = pd.to_numeric(df["VALTODAY"], errors="coerce")
    return df[(df["VALTODAY"] > 0) & (df["LAST"].notna())].reset_index(drop=True)

def fetch_candles(ticker, days_back=60, interval=60):
    till = datetime.now(MSK_TZ)
    since = till - timedelta(days=days_back)
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
    all_rows, start, columns = [], 0, None

    while True:
        params = {"from": since.strftime("%Y-%m-%d %H:%M:%S"), "till": till.strftime("%Y-%m-%d %H:%M:%S"), "interval": interval, "start": start, "iss.meta": "off"}
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200: break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None: columns = js.get("columns", [])
        if not rows: break
        all_rows.extend(rows)
        if len(rows) < 500: break
        start += len(rows)

    if not all_rows: return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "volume": "Volume", "begin": "Date", "end": "DateEnd"})
    df = df[["Date", "DateEnd", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)

    now_msk = pd.Timestamp.now(tz=MSK_TZ).tz_localize(None)
    if len(df) > 0 and df["DateEnd"].iloc[-1] > now_msk:
        df = df.iloc[:-1]

    return df.drop(columns=["DateEnd"]).reset_index(drop=True)

def add_indicators(df):
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = -delta.clip(upper=0).ewm(alpha=1/14, adjust=False).mean()
    df["RSI"] = 100 - (100 / (1 + gain / loss))
    df["EMA50_Slope"] = (df["EMA50"] - df["EMA50"].shift(5)) / df["EMA50"].shift(5) * 100
    return df

# ==============================================================================
# СКАНИРОВАНИЕ (СТРОГИЙ КОД БЕЗ ПОГРЕШНОСТЕЙ)
# ==============================================================================
print("📊 Загружаем живой снимок рынка с Мосбиржи...")
stocks = get_market_snapshot()
proven_stocks_df = stocks[stocks["SECID"].isin(PROVEN_STOCKS.keys())].reset_index(drop=True)

print(f"✅ Сканируем {len(proven_stocks_df)} акций из 'Белого Списка'...\n")

active_signals = []
watchlist = []

for index, row in tqdm(proven_stocks_df.iterrows(), total=len(proven_stocks_df), desc="Анализ сигналов"):
    ticker = row["SECID"]
    live_price = row["LAST"]
    cfg = PROVEN_STOCKS[ticker]

    try:
        df = fetch_candles(ticker, days_back=DAYS_BACK, interval=INTERVAL)
        if df.empty or len(df) < 200: continue

        df = add_indicators(df)

        c_now = df.iloc[-1]   # Последняя ЗАКРЫТАЯ свеча
        c_prev = df.iloc[-2]  # Предыдущая закрытая свеча

        # 1. Глобальный тренд
        is_uptrend = c_now["EMA50"] > c_now["EMA200"]
        is_slope_ok = c_now["EMA50_Slope"] >= 0.01

        # 2. ЖЕСТКОЕ ФИЗИЧЕСКОЕ КАСАНИЕ EMA50 (БЕЗ ВСЯКИХ МНОЖИТЕЛЕЙ!)
        # Цена предыдущей свечи ОБЯЗАНА была коснуться или пробить EMA50 вниз
        strict_touch = (c_prev["Low"] <= c_prev["EMA50"]) or (c_prev["Close"] <= c_prev["EMA50"])

        # 3. СВЕЖИЙ РАЗВОРОТ: На этой свече цена закрылась ВЫШЕ EMA50 и свеча ЗЕЛЕНАЯ
        strict_bounce = (c_now["Close"] > c_now["EMA50"]) and (c_now["Close"] > c_now["Open"])

        # 4. Фильтр перепроданности
        pass_rsi = (c_prev["RSI"] <= 55) or (c_now["RSI"] <= 55)

        # СИГНАЛ ТОЛЬКО ЕСЛИ БЫЛО ФИЗИЧЕСКОЕ КАСАНИЕ КРАСНОЙ ЛИНИИ
        is_buy_signal = is_uptrend and is_slope_ok and strict_touch and strict_bounce and pass_rsi

        if is_buy_signal:
            entry = live_price
            sl = min(c_prev["Low"], c_now["Low"])
            risk = entry - sl
            if risk <= 0: risk = entry * 0.005; sl = entry - risk

            tp = entry + (risk * cfg["RR"])
            risk_pct = (risk / entry) * 100

            active_signals.append({
                "Ticker": ticker, "Name": cfg["Name"], "Entry": entry, "SL": sl, "TP": tp,
                "Risk": risk, "RiskPct": risk_pct, "Config": cfg
            })

        # В список наблюдения берем только те, у которых цена реально близко к EMA50 (не дальше 0.8%)
        elif is_uptrend and (abs(live_price - c_now["EMA50"]) / c_now["EMA50"] * 100 <= 0.8):
            watchlist.append({
                "Тикер": ticker, "Компания": cfg["Name"], "Цена": smart_round(live_price),
                "EMA50": smart_round(c_now["EMA50"]),
                "Статус": "⏳ Подходит к EMA50 (Ждем касания красной линии)"
            })

        time.sleep(0.05)

    except Exception as e:
        continue

# ==============================================================================
# ВЫВОД
# ==============================================================================
print("\n" + "="*80)

if active_signals:
    print(f"🔥 НАЙДЕНО ГОТОВЫХ СИГНАЛОВ НА ПОКУПКУ: {len(active_signals)}")
    print("="*80)

    for sig in active_signals:
        c = sig["Config"]
        e, sl, tp, r = sig["Entry"], sig["SL"], sig["TP"], sig["Risk"]

        print(f"\n🟢 [BUY SIGNAL] {sig['Ticker']} ({sig['Name']})")
        print(f"   ├─ 📍 ВХОД (Цена прямо сейчас): {smart_round(e)} руб.")
        print(f"   ├─ 🛡️ СТОП-ЛОСС:               {smart_round(sl)} руб. (Риск: {smart_round(r)} руб. / {sig['RiskPct']:.2f}%)")
        print(f"   └─ 🎯 ТЕЙК-ПРОФИТ:             {smart_round(tp)} руб. (Соотношение 1:{c['RR']})")
        print(f"   ------------------------------------------------------------")
        print(f"   📋 ИНСТРУКЦИЯ ПО УПРАВЛЕНИЮ:")
        print(f"      • Прессет: {c['Mode']}")
        print(f"      • Совет:   {c['Note']}")
        if c["BE"] > 0:
            print(f"      • Безубыток: перевести стоп на {smart_round(e)} руб. при достижении цены {smart_round(e + r * c['BE'])} руб.")
        if c["Trail"] > 0:
            print(f"      • Трейлинг:  тянуть стоп вслед за ценой на расстоянии {smart_round(r * c['Trail'])} руб. от пика.")
        print("="*80)
else:
    print("❌ Свежих сигналов нет. Все бумаги либо далеко от EMA50, либо не касались красной линии.")

if watchlist:
    print("\n👁️ АКЦИИ В ЗОНЕ НАБЛЮДЕНИЯ (Близко к красной линии EMA50):")
    print("-" * 80)
    wl_df = pd.DataFrame(watchlist)
    print(wl_df.to_string(index=False))

print("="*80)

📊 Загружаем живой снимок рынка с Мосбиржи...
✅ Сканируем 10 акций из 'Белого Списка'...



Анализ сигналов: 100%|██████████| 10/10 [00:15<00:00,  1.60s/it]


🔥 НАЙДЕНО ГОТОВЫХ СИГНАЛОВ НА ПОКУПКУ: 2

🟢 [BUY SIGNAL] LSNGP (РСетиЛЭ-п)
   ├─ 📍 ВХОД (Цена прямо сейчас): 361.45 руб.
   ├─ 🛡️ СТОП-ЛОСС:               359.55 руб. (Риск: 1.9 руб. / 0.53%)
   └─ 🎯 ТЕЙК-ПРОФИТ:             365.25 руб. (Соотношение 1:2.0)
   ------------------------------------------------------------
   📋 ИНСТРУКЦИЯ ПО УПРАВЛЕНИЮ:
      • Прессет: 3. Умеренный
      • Совет:   Тейк 1:2. Б/У при +1R, Трейлинг 1.0R
      • Безубыток: перевести стоп на 361.45 руб. при достижении цены 363.35 руб.
      • Трейлинг:  тянуть стоп вслед за ценой на расстоянии 1.9 руб. от пика.

🟢 [BUY SIGNAL] YDEX (Яндекс)
   ├─ 📍 ВХОД (Цена прямо сейчас): 4004.5 руб.
   ├─ 🛡️ СТОП-ЛОСС:               3980.5 руб. (Риск: 24.0 руб. / 0.60%)
   └─ 🎯 ТЕЙК-ПРОФИТ:             4076.5 руб. (Соотношение 1:3.0)
   ------------------------------------------------------------
   📋 ИНСТРУКЦИЯ ПО УПРАВЛЕНИЮ:
      • Прессет: 1. Макс Профит
      • Совет:   Жесткий тейк 1:3. Не трогать стоп руками!

👁️ А

In [ ]:
High +5%

In [1]:
# ==============================================================================
# ВЕЧЕРНИЙ СКАНИРОВЩИК СИГНАЛОВ В ШОРТ (По стратегии "После Пампа +5%")
# ==============================================================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

TOP_LIQUID_COUNT = 130   # Сканируем 130 ликвидных акций (маржинальные бумаги)
PUMP_THRESHOLD_PCT = 5.0 # Ищем росты от +5% за сегодня
SL_BUFFER_PCT = 1.5      # Запас стоп-лосса (+1.5% над хаем пампа)

# БАЗА ЗНАНИЙ ИЗ 2-ЛЕТНЕГО БЭКТЕСТА
# 1. Послушные акции (Высокий винрейт падения)
HIGH_PROB_STOCKS = {
    "VTBR": "ВТБ (Историческая вероятность падения: 100%)",
    "NVTK": "Новатэк (Историческая вероятность падения: 86.7%)",
    "BSPB": "Банк СПб (Историческая вероятность падения: 85.7%)",
    "PIKK": "ПИК (Историческая вероятность падения: 88.9%)",
    "GMKN": "Норникель (Историческая вероятность падения: 80.0%)",
    "CHMF": "Северсталь (Историческая вероятность падения: 81.2%)",
    "AFLT": "Аэрофлот (Историческая вероятность падения: 80.0%)",
    "DIAS": "Диасофт (Историческая вероятность падения: 80.0%)",
    "SIBN": "Газпромнефть (Историческая вероятность падения: 83.3%)",
    "EUTR": "ЕвроТранс (Историческая вероятность падения: 75.0%)",
    "SVAV": "Соллерс (Историческая вероятность падения: 80.0%)",
    "ELMT": "Элемент (Историческая вероятность падения: 100%)",
    "VRSB": "ТНС энерго Воронеж (Историческая вероятность падения: 100%)",
}

# 2. Опасные акции (Продолжают ралли, ШОРТИТЬ ЗАПРЕЩЕНО)
DANGER_STOCKS = ["YDEX", "MSNG", "LSNGP", "IRAO", "POSI", "MTLR", "LKOH", "BANE", "PRMD", "DATA"]

def smart_round(val):
    if val >= 100: return round(val, 2)
    elif val >= 1: return round(val, 3)
    else: return round(val, 5)

def get_liquid_tickers(top_n):
    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {"iss.meta": "off", "iss.only": "securities,marketdata", "securities.columns": "SECID,SHORTNAME", "marketdata.columns": "SECID,VALTODAY"}

    for _ in range(3):
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 200:
                js = r.json()
                sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
                md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])
                df = sec.merge(md, on="SECID", how="left")
                df["VALTODAY"] = pd.to_numeric(df["VALTODAY"], errors="coerce").fillna(0)

                ignore_list = ["LQDT", "AKMM", "SBMM", "BCSB", "TPAY"]
                df = df[~df["SECID"].isin(ignore_list)]
                return df.sort_values("VALTODAY", ascending=False).head(top_n).reset_index(drop=True)
        except: time.sleep(1)
    return pd.DataFrame()

def fetch_recent_daily_candles(ticker, days_back=15):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"

    params = {"from": since.strftime("%Y-%m-%d"), "till": till.strftime("%Y-%m-%d"), "interval": 24, "iss.meta": "off"}

    for _ in range(3):
        try:
            r = requests.get(url, params=params, timeout=10)
            if r.status_code == 200:
                js = r.json().get("candles", {})
                rows = js.get("data", [])
                cols = js.get("columns", [])
                if not rows: return pd.DataFrame()

                df = pd.DataFrame(rows, columns=cols)
                df["begin"] = pd.to_datetime(df["begin"])
                df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "volume": "Volume", "begin": "Date"})
                return df[["Date", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date").reset_index(drop=True)
            time.sleep(1)
        except: time.sleep(1)
    return pd.DataFrame()

# ==========================================
# ПОИСК СИГНАЛОВ
# ==========================================
print("📊 Сканируем Мосбиржу на предмет сегодняшних пампов (рост >= +5%)...")
stocks_df = get_liquid_tickers(TOP_LIQUID_COUNT)

active_short_signals = []

for idx, row in tqdm(stocks_df.iterrows(), total=len(stocks_df), desc="Анализ Дневных свечей"):
    ticker = row["SECID"]
    name = row["SHORTNAME"]

    df = fetch_recent_daily_candles(ticker, days_back=15)
    if df.empty or len(df) < 3: continue

    # Берем последнюю закрытую дневную свечу (Сегодняшний день T)
    today = df.iloc[-1]
    yesterday = df.iloc[-2]

    # Расчет дневного роста сегодня относительно закрытия вчера
    daily_gain_pct = (today["Close"] - yesterday["Close"]) / yesterday["Close"] * 100

    # Если сегодня был ПАМП на +5% и более
    if daily_gain_pct >= PUMP_THRESHOLD_PCT:

        # Расчет Стоп-Лосса (Вчерашний High + 1.5% запаса)
        sl_price = today["High"] * (1 + SL_BUFFER_PCT / 100)
        risk_from_close_pct = (sl_price - today["Close"]) / today["Close"] * 100

        # Проверяем фильтры безопасности
        is_danger = ticker in DANGER_STOCKS
        is_top_obedient = ticker in HIGH_PROB_STOCKS

        note = HIGH_PROB_STOCKS.get(ticker, "Обычная акция (Стандартный риск)")
        if is_danger:
            note = "⚠️ ОПАСНОСТЬ! В бэктесте акция продолжала расти. Шортить НЕ РЕКОМЕНДУЕТСЯ!"

        active_short_signals.append({
            "Ticker": ticker, "Name": name, "Date": today["Date"].strftime("%Y-%m-%d"),
            "GainPct": round(daily_gain_pct, 2), "TodayClose": today["Close"],
            "TodayHigh": today["High"], "SL_Price": sl_price,
            "RiskPct": round(risk_from_close_pct, 2), "IsDanger": is_danger,
            "IsTop": is_top_obedient, "Note": note
        })

    time.sleep(0.02)

# ==========================================
# ВЫВОД КАРТОЧЕК СИГНАЛОВ
# ==========================================
print("\n" + "="*80)

if not active_short_signals:
    print("❌ Сегодня на Мосбирже не было ни одного пампа на +5% и более.")
    print("   Сигналов в шорт на завтра НЕТ. Отдыхаем!")
else:
    print(f"🔥 НАЙДЕНО АКЦИЙ С ПАМПОМ +5% СЕГОДНЯ: {len(active_short_signals)}")
    print("="*80)

    for sig in active_short_signals:
        t = sig["Ticker"]
        status_icon = "🔴" if not sig["IsDanger"] else "❌"

        print(f"\n{status_icon} [SHORT SIGNAL ON TOMORROW] {t} ({sig['Name']})")
        print(f"   ├─ 📈 Дневной рост сегодня:  +{sig['GainPct']}% (Цена закрытия: {smart_round(sig['TodayClose'])} руб.)")
        print(f"   ├─ 📍 ТОЧКА ВХОДА:           Завтра (на открытии в 10:00 МСК по рынку)")
        print(f"   ├─ 🛡️ СТОП-ЛОСС:             {smart_round(sig['SL_Price'])} руб. (High дня {smart_round(sig['TodayHigh'])} + 1.5% запас)")
        print(f"   ├─ 🎯 ТЕЙК-ПРОФИТ:           НЕ СТАВИМ! (Выходим в конце дня)")
        print(f"   └─ ⏰ ВЫХОД ИЗ СДЕЛКИ:        Завтра в 18:40 МСК (Закрываем шорт по рынку)")
        print(f"   ------------------------------------------------------------")
        print(f"   📋 АНАЛИТИКА БЭКТЕСТА: {sig['Note']}")
        print("="*80)

📊 Сканируем Мосбиржу на предмет сегодняшних пампов (рост >= +5%)...


Анализ Дневных свечей: 100%|██████████| 130/130 [01:50<00:00,  1.18it/s]


❌ Сегодня на Мосбирже не было ни одного пампа на +5% и более.
   Сигналов в шорт на завтра НЕТ. Отдыхаем!
